# 01 · Exploratory data analysis

**Goal:** get familiar with every provided source and extract the facts that drive modelling and
business choices. Findings are summarised at the end and in `docs/eda_findings.md`
(outputs are stripped on commit, so the written summary is the reference).

All loaders live in `src/schneider_forecast/data.py`, so other notebooks reuse the same parsing.

Sections
1. Inventory and forecast origin
2. Price history and the 2021–23 spike
3. Steel: the free proxies for the empty licensed series
4. Co-movement → how to group materials
5. Predictability: volatility, naive baseline error, leading indicators
6. Epoxy, energy, FX
7. BOM: from €/t to € per unit
8. RMU price panel
9. Competitor benchmark
10. Supplier financials (anonymised)
11. Nominal vs real
12. Summary of findings and open decisions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import eda_loaders as d

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})

panel = d.load_monthly_panel().loc["2016-11":]
rm_cat = d.load_catalog("rm")
energy_cat = d.load_catalog("energy")

## 1. Inventory and forecast origin

Which series exist, from when to when. The **forecast origin** (slide 17) is `2026-09`: the last
month for which *all* RM series are observed. Some sources run one month longer (steel, epoxy,
FX, some energy) and stainless steel stops one month earlier (Aug 2026).

In [ ]:
rm = d.load_rm_prices()
steel = d.load_steel()
sources = {
    "RM prices (S/D)": rm.drop(columns=[c for c in rm if c.startswith("L")]),
    "Energy (E/ED)": d.load_energy_prices(),
    "Steel MEPS": d.load_steel_meps(),
    "Steel CRU flat": d.load_steel_cru_flat(),
    "Steel CRU stainless": d.load_steel_cru_stainless(),
    "Epoxy": d.load_epoxy(),
    "LME Cu/Al": d.load_lme(),
}
inventory = pd.DataFrame(
    {
        name: {
            "n_series": df.shape[1],
            "first_month": df.index.min().strftime("%Y-%m"),
            "last_month_any": df.index.max().strftime("%Y-%m"),
            "last_month_all": df.dropna(how="any", axis=1).index.max().strftime("%Y-%m")
            if df.dropna(how="any", axis=1).shape[1]
            else "-",
            "missing_%": round(100 * df.isna().mean().mean(), 1),
        }
        for name, df in sources.items()
    }
).T
inventory

In [ ]:
# Last observed month of every candidate forecast target
targets = [
    "S01",
    "S02",
    "S03",
    "S04",
    "S05",
    "S06",
    "S07",
    "S08",
    "S09",
    "S10",
    "S11",
    "S12",
    "S13",
    "D01",
    "D02",
    "L01_proxy",
    "L02_proxy",
    "L03_proxy",
    "epoxy_resin",
]
last_obs = panel[targets].apply(lambda c: c.last_valid_index().strftime("%Y-%m"))
last_obs.groupby(last_obs).apply(lambda s: ", ".join(s.index)).rename("series").to_frame()

In [ ]:
# Availability heatmap (2016-11 -> 2026-10): white = missing
avail = panel[targets].notna().T.astype(int)
fig, ax = plt.subplots(figsize=(11, 5))
ax.imshow(avail, aspect="auto", cmap="Greens", interpolation="none")
ax.set_yticks(range(len(targets)), targets)
ticks = range(0, avail.shape[1], 12)
ax.set_xticks(ticks, [avail.columns[i].strftime("%Y-%m") for i in ticks])
ax.set_title("Data availability per target series")
plt.show()

### 1b. Anomaly scan
A "spike" = a month whose log return is extreme (> 4 robust standard deviations, MAD-based) **and**
is mostly reversed the next month. Real market shocks persist; one-month spikes are usually typos.

In [ ]:
scan_cols = (
    targets
    + ["benzene", "acetone", "epichlorohydrin", "epoxy_other_costs"]
    + list(d.load_energy_prices().columns.drop("E23"))
)
ret = np.log(panel[scan_cols]).diff()
mad = (ret - ret.median()).abs().median() * 1.4826
z = (ret - ret.median()) / mad
reversal = -ret.shift(-1) / ret  # 1 = fully reversed next month
flags = (z.abs() > 4) & (reversal > 0.5)
hits = flags.stack()
hits = hits[hits].index  # (month, series) pairs that are flagged
anomalies = pd.DataFrame(
    {
        "month": hits.get_level_values(0),
        "series": hits.get_level_values(1),
        "robust_z": [z.loc[m, s] for m, s in hits],
        "reversed_share": [reversal.loc[m, s] for m, s in hits],
    }
)
anomalies["value"] = [
    panel.loc[m, s] for m, s in zip(anomalies.month, anomalies.series, strict=True)
]
anomalies["prev"] = [
    panel[s].shift(1).loc[m] for m, s in zip(anomalies.month, anomalies.series, strict=True)
]
anomalies.sort_values("robust_z", key=abs, ascending=False).round(2)

**Reading.** The clear data error is **epichlorohydrin in May 2026: 257 €/t instead of ~2,500**
(a missing digit), which also drags the computed epoxy resin price down that month (3,593 vs ~4,900).
Proposed treatment: set both to NaN for May 2026 and log it in `docs/data_transformations.md`.
The other flags are real market events, not errors (Japan power price spike Jan 2021, WTI oil
crash Apr 2020, ...): keep them.

## 2. Price history and the 2021–23 spike

All series rebased to **2019 average = 100** so prices (USD/t, EUR/t) and indices are comparable.

In [ ]:
def rebase(df, base="2019"):
    return 100 * df / df.loc[base].mean()


groups = {
    "Non-ferrous metals (EUR)": ["D01", "D02", "S03", "S05"],
    "Steel market prices (EUR/t)": ["L01_proxy", "L02_proxy", "L03_proxy", "S04"],
    "EU producer price indices": ["S07", "S08", "S09", "S10", "S11"],
    "Insulation / plastics": ["epoxy_resin", "S11", "S13"],
}
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, (title, cols) in zip(axes.flat, groups.items(), strict=True):
    rebase(panel[cols]).plot(ax=ax, lw=1.4)
    ax.axvspan("2021-01", "2023-06", color="grey", alpha=0.12)
    ax.set_title(title)
    ax.axhline(100, color="k", lw=0.6)
plt.tight_layout()
plt.show()

In [ ]:
x = panel[targets]
spike = pd.DataFrame(
    {
        "peak_month_2020_24": x.loc["2020":"2024"].idxmax().dt.strftime("%Y-%m"),
        "peak_vs_2019": x.loc["2020":"2024"].max() / x.loc["2019"].mean() - 1,
        "latest_vs_2019": x.ffill().loc["2026-09"].iloc[0] / x.loc["2019"].mean() - 1,
        "yoy_to_latest": x.ffill().loc["2026-09"].iloc[0] / x.ffill().loc["2025-09"].iloc[0] - 1,
    }
)
spike.style.format({c: "{:+.0%}" for c in spike.columns[1:]})

**Reading.** The spike hits every family but with different timing: iron ore peaks first (mid-2021),
LME metals in Mar 2022 (Ukraine invasion), steel market prices in Apr–May 2022, plastics / epoxy
and the PPIs mid-2022 (they lag the markets). Steel and nickel have largely mean-reverted; **copper
has not: it is at an all-time high in 2026 (~+45 % year on year)**. Any model trained on 2017–2026
must cope with this regime change: options are (a) keep it and let bands widen, (b) down-weight it,
(c) add a regime/volatility feature. Removing it would make P10/P90 overconfident.

## 3. Steel: free proxies for the empty licensed series

L01 (CR coil), L02 (stainless 304) and L03 (HDG coil) are empty on purpose. The steel workbook gives
real €/t prices from two publishers. Default proxies in `data.STEEL_PROXIES`:
`L01 ← MEPS CR coil Germany`, `L02 ← CRU 304 Germany (base + alloy surcharge)`,
`L03 ← MEPS HDG coil Germany`. Checks below: do MEPS and CRU agree, how do countries differ,
and how does the market price relate to the Eurostat PPI (S09/S10)?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cr_cols = [
    "MEPS_CR_coil_Germany",
    "CRU_CR_coil_Germany",
    "MEPS_CR_coil_France",
    "MEPS_CR_coil_Italy",
]
steel[cr_cols].loc["2016":].plot(ax=axes[0], lw=1.2)
axes[0].set_title("CR coil, EUR/t: publishers and countries")
ss_cols = ["CRU_SS304_Germany_base", "CRU_SS304_Germany_alloy", "CRU_SS304_Germany_total"]
steel[ss_cols].loc["2016":].plot(ax=axes[1], lw=1.2)
axes[1].set_title("Stainless 304 Germany: base + alloy surcharge")
rebase(panel[["L01_proxy", "S09", "S10", "S12"]]).plot(ax=axes[2], lw=1.2)
axes[2].set_title("Market price vs PPI (2019 = 100)")
plt.tight_layout()
plt.show()

gap = (steel["CRU_CR_coil_Germany"] / steel["MEPS_CR_coil_Germany"] - 1).loc["2016":]
print(f"CRU vs MEPS CR coil Germany: mean gap {gap.mean():+.1%}, abs mean {gap.abs().mean():.1%}")

In [ ]:
# Lead-lag on monthly log returns: does the market price lead the PPI?
r = np.log(panel).diff()
lead = pd.DataFrame(
    {
        "CR coil (t) -> PPI S09 (t+k)": {
            k: r["L01_proxy"].corr(r["S09"].shift(-k)) for k in range(0, 5)
        },
        "Iron ore S04 (t) -> CR coil (t+k)": {
            k: r["S04"].corr(r["L01_proxy"].shift(-k)) for k in range(0, 5)
        },
        "Nickel S03 (t) -> SS304 (t+k)": {
            k: r["S03"].corr(r["L02_proxy"].shift(-k)) for k in range(0, 5)
        },
    }
)
lead.index.name = "lag k (months)"
lead.round(2)

**Reading.** MEPS and CRU tell the same story (use one consistently, keep the other as a check).
The stainless price is mostly **alloy surcharge** (nickel-driven), the base price is small.
The PPI S09 is a smoothed, ~1-month-lagged version of the market price, so it understates
swings: use real €/t proxies for cost, PPIs as features. Iron ore leads CR coil by 1–2 months.

## 4. Co-movement → how to group materials

Correlation of **monthly log returns** (levels are trending and would show spurious correlation).

In [ ]:
cols = [
    "D01",
    "D02",
    "S03",
    "S05",
    "S04",
    "L01_proxy",
    "L03_proxy",
    "L02_proxy",
    "S09",
    "S10",
    "S11",
    "S13",
    "epoxy_resin",
    "E01",
    "ED02",
    "S06",
]
corr = np.log(panel[cols]).diff().loc["2017":].corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=90)
ax.set_yticks(range(len(cols)), cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.1f}", ha="center", va="center", fontsize=7)
fig.colorbar(im)
ax.set_title("Correlation of monthly log returns, 2017-2026")
plt.show()

**Reading.** Clear blocks support grouping materials by driver:
- **Non-ferrous** (copper, aluminium, zinc, nickel) move together.
- **Flat carbon steel** (CR and HDG coil ≈ 0.98) is one series in practice; stainless is related but
  distinct (nickel-driven); the EU steel PPIs form their own smoothed block.
- **Epoxy** and **FX** are nearly uncorrelated with everything at monthly frequency.

Suggested forecast groups: copper · aluminium · carbon/galvanised steel · stainless · plastics (PPI) ·
epoxy · FX. One model per group (shared features), then each target series is forecast within it.

## 5. Predictability

### 5a. How far do prices move in 3 / 6 / 9 months?
The standard deviation of the h-month log change is roughly the width a P10–P90 band needs
(P10–P90 ≈ ±1.28 sd). The mean absolute h-month change since 2020 is the **naive baseline error**
(forecast = last value) that every model must beat.

In [ ]:
lr = np.log(panel[targets])
vol = {}
for h in (3, 6, 9):
    ch = lr.shift(-h) - lr
    vol[(h, "sd of change")] = ch.std()
    vol[(h, "naive MAE (2020+)")] = ch.loc["2020":].abs().mean()
vol = pd.DataFrame(vol)
vol.style.format("{:.1%}").background_gradient(axis=None, cmap="Oranges")

**Reading.** Errors grow roughly with √h. Steel market prices (±20–25 % at 9 months) and copper
(±14–17 %) are the hard ones; PPIs and epoxy move less (they are smoothed). FX moves ~5 %.

### 5b. Leading indicators (first look)
Correlation between the **past 3-month change** of a driver and the **future h-month change** of a
target. Only uses information known at the forecast date. ⚠️ In-sample and overlapping windows:
this inflates correlations, so treat it as a shortlist to test in the backtest, not as proof.

In [ ]:
fut_targets = ["D01", "D02", "L01_proxy", "L02_proxy", "S09", "S11", "epoxy_resin"]
drivers = [
    "D01",
    "D02",
    "S03",
    "S04",
    "S05",
    "L01_proxy",
    "S09",
    "S11",
    "S13",
    "E01",
    "ED02",
    "E09",
    "S06",
    "benzene",
    "epichlorohydrin",
]
llr = np.log(panel)
past = llr - llr.shift(3)
for h in (3, 6):
    fut = llr.shift(-h) - llr
    m = pd.DataFrame({t: {dr: fut[t].corr(past[dr]) for dr in drivers} for t in fut_targets})
    display(
        m.style.format("{:.2f}")
        .background_gradient(axis=None, cmap="RdBu_r", vmin=-0.7, vmax=0.7)
        .set_caption(f"corr(past 3m change of driver [rows], future {h}m change of target [cols])")
    )

**Reading.**
- **Copper and aluminium look like random walks**: nothing (not even their own momentum) predicts
  them. Realistic goal there: honest bands, not beating naive by much.
- **Smoothed series (PPI S09/S11, epoxy) have momentum** and react to past metal/energy moves, so
  they are where a model can beat the naive baseline.
- **Steel market prices** react to past copper/iron ore/zinc moves with a lag.

In [ ]:
# The LME 3-month price as "the market's own forecast" for copper
lme = d.load_lme()
spread = lme["cu_3m_usd_per_t"] / lme["cu_cash_usd_per_t"] - 1
realised = np.log(lme["cu_cash_usd_per_t"]).shift(-3) - np.log(lme["cu_cash_usd_per_t"])
check = pd.concat([lme["cu_cash_eur_per_t"], panel["D01"]], axis=1).dropna()
diff = (check.iloc[:, 0] / check.iloc[:, 1] - 1).abs().mean()
print(f"LME file vs D01 (copper EUR/t): mean abs diff {diff:.2%}")
print(f"corr(3m-cash spread, realised 3m change): {spread.corr(realised):.2f}")

The LME file matches D01 almost exactly (good cross-check, and it adds Oct 2026). The futures
spread has no predictive power for the realised change, which again points to "random walk + band"
for copper.

## 6. Epoxy, energy and FX

In [ ]:
ep = d.load_epoxy()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ep.plot(ax=axes[0], lw=1.2)
axes[0].set_title("Epoxy resin and inputs, EUR/t")
rebase(panel[["ED02", "E09", "E10", "E11", "E01"]]).plot(ax=axes[1], lw=1.2, logy=True)
axes[1].set_title("Energy (2019 = 100, log scale)")
plt.tight_layout()
plt.show()
ep.corr()["epoxy_resin"].round(2).to_frame("corr with resin (levels)")

In [ ]:
fx = panel[["S06", "E24"]].dropna()
ax = fx["S06"].plot(title="EUR/USD (S06)", lw=1.4)
plt.show()
print(f"EUR/USD range since 2016: {fx['S06'].min():.3f} - {fx['S06'].max():.3f}")

**Reading.** Epoxy resin is built from its inputs (≈ benzene + acetone + epichlorohydrin + other
costs), so it can be forecast from them, but its monthly moves are small and independent of metals.
The insulating material is this epoxy series. European gas and power went up
~15× in 2022: a strong cost-push driver for steel, aluminium and plastics, with lags.
EUR/USD is close to a random walk; convert USD prices to EUR with the last known rate and treat FX
as a separate risk in the buffer.

## 7. BOM: from €/t to € per unit

⚠️ **Two BOM versions disagree** : the CSV in `data.zip` and the
standalone `RM_Prices_XHEC_Challenge.xlsx`.

In [ ]:
b_csv, b_xlsx = d.load_bom("csv"), d.load_bom("xlsx")
cmp = b_csv[["product", "material_label_source", "material_family", "share_pct", "mass_kg"]].copy()
cmp["share_pct_xlsx"] = b_xlsx["share_pct"].values
cmp["mass_kg_xlsx"] = b_xlsx["mass_kg"].values
cmp["mass_diff_kg"] = cmp["mass_kg_xlsx"] - cmp["mass_kg"]
cmp

Material cost per unit = mass (kg) × price (€/kg). Priceable materials only: plastics have no € level
(index only) and "Other" has no composition. RM6 carbon steel has no mass in the BOM: we assume
45 % share of the implied total mass (stainless 55.8 kg / 18 % = 310 kg → **≈ 140 kg, assumption**).

In [ ]:
eur_per_kg = (
    pd.DataFrame(
        {
            "Carbon steel": panel["L01_proxy"],
            "Stainless steel": panel["L02_proxy"].ffill(),
            "Galvanised steel": panel["L03_proxy"],
            "Copper": panel["D01"],
            "Aluminium": panel["D02"],
        }
    ).loc["2017":"2026-09"]
    / 1000
)


def unit_cost(bom, product, rm6_carbon_steel_kg=0.45 * 55.8 / 0.18):
    rows = bom[(bom["product"] == product) & bom["mass_kg"].notna()]
    out = {
        r.material_family: eur_per_kg[r.material_family] * r.mass_kg
        for r in rows.itertuples()
        if r.material_family in eur_per_kg
    }
    if product == "RM6":
        out["Carbon steel"] = eur_per_kg["Carbon steel"] * rm6_carbon_steel_kg
    return pd.DataFrame(out)


cost = {p: unit_cost(b_csv, p) for p in ("RM6", "RM AirSeT")}
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (p, c) in zip(axes, cost.items(), strict=True):
    c.plot.area(ax=ax, lw=0)
    ax.set_title(f"{p}: priceable raw-material cost, EUR/unit")
plt.tight_layout()
plt.show()

latest = pd.DataFrame({p: c.loc["2026-09"].iloc[0] for p, c in cost.items()}).fillna(0)
latest.loc["TOTAL"] = latest.sum()
latest.round(0)

In [ ]:
total = pd.DataFrame({p: c.sum(axis=1) for p, c in cost.items()})
ax = total.plot(lw=1.6, title="Priceable RM cost per unit, EUR (CSV BOM, steel proxies)")
ax.axvspan("2021-01", "2023-06", color="grey", alpha=0.12)
plt.show()
(total["RM6"] - total["RM AirSeT"]).resample("YE").mean().round(0).rename(
    "RM6 minus AirSeT, EUR"
).to_frame().T

**Reading.**
- **RM6 cost is dominated by copper (~60 % of priceable cost in Sep 2026)**; RM AirSeT has *no copper
  line* and is ~50/50 carbon steel / stainless. The two products are exposed to different risks.
- With copper at a record and steel back down, the priceable RM cost of RM6 now **exceeds** RM AirSeT.
  This is the "RM price convergence" lever named on slide 15 for Goal B, to be stress-tested
  (BOM uncertainty, unpriced plastics/"Other", purchased components).
- Priceable RM (~€550–800) is far below the benchmark material should-cost (~€2,400 for RM6), which
  also includes purchased components. For pricing, model a **cost index** (% change) applied to the
  RM-driven share of cost, rather than trusting absolute € levels.

## 8. RMU price panel

Synthetic yearly selling prices 2017–2026 (EUR ex-VAT) per brand with P10/P50/P90. Eaton (ETN) is the
only SF6-free product. No 1F–4F split is present in this file despite the slides.

In [ ]:
rmu = d.load_rmu_panel()
p50 = rmu.pivot(index="year", columns="brand", values="p50")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
p50.plot(ax=axes[0], marker="o", lw=1.4)
axes[0].set_title("RMU selling price P50, EUR")
se = rmu[rmu.brand_id == "SE"].set_index("year")
axes[1].fill_between(se.index, se.p10, se.p90, alpha=0.25, label="SE P10-P90")
axes[1].plot(se.index, se.p50, marker="o", label="SE P50")
axes[1].set_title("Schneider Electric price band")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
yearly_cost = total.groupby(total.index.year).mean()
view = pd.DataFrame(
    {
        "SE price P50": se["p50"],
        "SE price YoY": se["p50"].pct_change(),
        "RM6 priceable RM cost": yearly_cost["RM6"],
        "RM cost YoY": yearly_cost["RM6"].pct_change(),
        "price / RM cost": se["p50"] / yearly_cost["RM6"],
        "Eaton (SF6-free) vs SE": p50["Eaton"] / p50["Schneider Electric"] - 1,
    }
)
view.style.format(
    {
        "SE price P50": "{:,.0f}",
        "RM6 priceable RM cost": "{:,.0f}",
        "SE price YoY": "{:+.0%}",
        "RM cost YoY": "{:+.0%}",
        "price / RM cost": "{:.1f}",
        "Eaton (SF6-free) vs SE": "{:+.0%}",
    }
)

**Reading.**
- **Prices lag costs.** In 2021 the RM cost jumped ~+60 % while the selling price moved +7 %: the
  price/cost multiple collapsed from ~17 to ~11 and only recovered by 2024–25. That is exactly the
  margin squeeze the challenge wants to prevent with forecast-based quoting.
- **2026: +40 % price jump for all SF6 brands** (synthetic), much larger than our RM cost move.
- **The SF6-free premium disappears**: Eaton was ~+38 % above Schneider in 2017 and is *below* it in
  2026. SF6-free pricing looks more stable. Useful for the Goal B story.
- The "price = predicted RM cost × historical margin" idea must choose *which*
  history: the multiple is not stable (11–17).

## 9. Competitor benchmark (provisional should-cost teardown)

In [ ]:
bm = d.load_benchmark()
cols = [
    "brand",
    "insulation",
    "made_in",
    "should_cost_total",
    "should_cost_material",
    "material_share",
    "list_price_eur",
    "mass_g",
]
bm[cols].style.format(
    {
        "should_cost_total": "{:,.0f}",
        "should_cost_material": "{:,.0f}",
        "material_share": "{:.0%}",
        "list_price_eur": "{:,.0f}",
        "mass_g": "{:,.0f}",
    },
    na_rep="-",
)

**Reading.** Material is ~80 % of should-cost for every credible row → RM forecasting is the right
lever. SGC (€19) is broken. Only Eaton Xiria is SF6-free. Only one real list price (Siemens €15,530,
~4.7× should-cost): consistent with the RMU panel's €10–14k range.

## 10. Supplier financials (anonymised)

Labels A–I follow file order; real names stay out of code and slides (NDA). Latest available year.

In [ ]:
fin = d.load_supplier_financials()
wanted = [
    "Revenue % Growth",
    "EBITDA Margin",
    "Total Debt to Equity",
    "Current Ratio",
    "Interest Coverage",
    "Total Revenue",
    "Total Assets",
]
latest = fin[fin.metric.isin(wanted)].sort_values("year").groupby(["supplier", "metric"]).last()
snap = latest["value"].unstack("metric")
snap["latest_year"] = latest["year"].unstack("metric").max(axis=1)
snap

**Reading.** Coverage is uneven (some suppliers have one year, some 20). Usable as a simple
supplier-risk score (liquidity, leverage, margin) feeding the risk buffer, not as a forecast input.

## 11. Nominal vs real

All provided prices are **nominal** (as published by LME, Eurostat, MEPS, CRU). The data contains no
consumer price index, so a real-terms view needs an external deflator (e.g. Eurostat HICP, euro area,
into `data/external/`). Schneider said this is optional extra insight. Forecast and submit in
nominal terms; use real terms only for the "is copper really at a record?" story.

## 12. Summary of findings and open decisions

| # | Finding | Consequence |
|---|---|---|
| 1 | Common forecast origin is **2026-09**; stainless stops 2026-08, steel/epoxy/FX run to 2026-10 | Nowcast stainless one month; decide whether to use Oct values (must be known at origin) |
| 2 | 2021–23 spike in all families, different timing; copper at a new record in 2026 | Keep spike in training, widen bands; test down-weighting in backtest |
| 3 | Strong blocks: non-ferrous, flat steel, PPIs; epoxy and FX independent | One model per group |
| 4 | Copper / aluminium ≈ random walk | Expect small gains vs naive; focus on calibrated bands |
| 5 | PPIs and epoxy are smoothed with momentum; steel lags iron ore / metals | This is where features/models can beat naive |
| 6 | 9-month moves: steel ±20–25 %, copper ±15 %, PPIs ±10 % | Size of P10/P90 and of the risk buffer |
| 7 | Two BOM versions disagree; RM6 carbon-steel mass missing | Run cost both ways until Schneider confirms |
| 8 | RM6 cost is copper-driven, AirSeT steel-driven; RM6 priceable cost now > AirSeT | Key Goal B argument (RM convergence) |
| 9 | Prices lag costs (multiple 17 → 11 in 2021); SF6-free premium vanished by 2026 | Quote rule / indexation; Goal B pricing |
| 10 | All data nominal, no deflator provided | Optional HICP from Eurostat |
| 11 | Data error: epichlorohydrin May 2026 = 257 (×10 too low) → epoxy resin dip | Set to NaN, log the fix |